# Lesson 3 · Similarity search: finding a text's nearest neighbours

<a href="https://colab.research.google.com/github/Mary-Vadi/rag/blob/main/lessons/03_similarity_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In this lesson you will**

- find the most similar texts in a pool of thousands with a single matrix multiplication
- do the same with **FAISS**, the library used in real retrieval systems
- look at a text's neighbours and ask the key question for the rest of the course: *do neighbours share a label?*

**Runtime:** a CPU works; a GPU makes the embedding step faster.

## 1. The problem

We have a **pool** of labelled texts and a new **query** text. We want the `k` pool texts that are most similar to the query: its **nearest neighbours**.

Thanks to lesson 2, "most similar" has a precise meaning: the highest cosine similarity between embeddings. And because our embeddings have length 1, cosine similarity is just a dot product.

## 2. Setup

In [ ]:
# Setup: run this cell first.
# In Colab it downloads the course code and installs the extra libraries.
# On your own computer (from the lessons/ folder) it just makes the code importable.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/rag"):
        !git clone -q https://github.com/Mary-Vadi/rag.git /content/rag
        !pip install -q -r /content/rag/requirements.txt
    %cd /content/rag
elif os.path.basename(os.getcwd()) == "lessons":
    %cd ..
sys.path.insert(0, os.getcwd())

In [ ]:
import textwrap
import time

import faiss
import numpy as np

from ragdetect.data import LABEL_NAMES, load_mage
from ragdetect.embeddings import Embedder

## 3. Build a pool and some queries

The pool comes from MAGE's **train** split and the queries from its **test** split. The two splits never share texts, so a query can never "find itself" in the pool, which would be cheating.

MAGE's train file is about 400 MB, so the first download takes a little while.

In [ ]:
pool = load_mage("train", n=5000)
queries = load_mage("test", n=200)

embedder = Embedder()   # all-MiniLM-L6-v2, as in lesson 2
pool_vectors = embedder.encode(pool["text"], show_progress=True)
query_vectors = embedder.encode(queries["text"], show_progress=True)
print("pool:", pool_vectors.shape, " queries:", query_vectors.shape)

## 4. Brute force: compare the query with everything

The simplest method is to compute the similarity between every query and every pool text, then keep the top `k`. With unit-length vectors, **one matrix multiplication** gives all 200 × 5,000 similarities at once:

```
query_vectors  (200, 384)   @   pool_vectors.T  (384, 5000)   =   scores  (200, 5000)
```

Row `i` of `scores` holds query `i`'s similarity to every pool text.

In [ ]:
k = 5

start = time.perf_counter()
scores = query_vectors @ pool_vectors.T
top_k = np.argsort(-scores, axis=1)[:, :k]     # positions of the k highest scores, best first
top_scores = np.take_along_axis(scores, top_k, axis=1)
brute_seconds = time.perf_counter() - start

print("scores:", scores.shape, " top_k:", top_k.shape)
print(f"brute force took {brute_seconds * 1000:.1f} ms")
print("query 0's neighbours:", top_k[0], "with similarities", top_scores[0].round(3))

## 5. The same thing with FAISS

[FAISS](https://github.com/facebookresearch/faiss) is a library from Meta for searching large collections of vectors. You put the pool vectors into an **index** once, then search it as often as you like.

We use `IndexFlatIP`: "Flat" means it compares the query with every vector (exact, just like our brute force), and "IP" means it scores by **inner product**, another name for the dot product.

In [ ]:
index = faiss.IndexFlatIP(pool_vectors.shape[1])
index.add(pool_vectors)
print("vectors in the index:", index.ntotal)

start = time.perf_counter()
faiss_scores, faiss_top_k = index.search(query_vectors, k)
faiss_seconds = time.perf_counter() - start

print(f"FAISS took {faiss_seconds * 1000:.1f} ms")
print("same neighbours as brute force:", f"{(faiss_top_k == top_k).mean():.1%}")

The two methods should agree on (almost) every neighbour. Tiny disagreements can only happen when two pool texts are tied, or within a rounding error of each other.

**Why bother with FAISS, then?** For a pool of 5,000 texts, brute force in NumPy is perfectly fine. But FAISS also offers *approximate* indexes (such as `IndexIVFFlat` and `IndexHNSWFlat`) that search millions or billions of vectors in milliseconds by skipping most of the comparisons, at the cost of occasionally missing a true neighbour. Every large RAG system relies on this kind of index. Starting with an exact index means we can switch later without changing any other code.

In `ragdetect/retrieval.py`, `NeighbourIndex` wraps exactly these steps (it also normalises the vectors, just in case).

In [ ]:
from ragdetect.retrieval import NeighbourIndex

neighbour_index = NeighbourIndex(pool_vectors)
sims, positions = neighbour_index.search(query_vectors, k)
print("same neighbours as the raw FAISS search:", f"{(positions == faiss_top_k).mean():.1%}")

## 6. Meet the neighbours

Numbers are one thing, but let's read some actual texts. For a query, print its label and the start of its text, then the same for its neighbours.

In [ ]:
def show_neighbours(i, k=5, width=250):
    query = queries.iloc[i]
    print(f"QUERY  [{LABEL_NAMES[query.label]} | {query.source}]")
    print(textwrap.fill(textwrap.shorten(query.text, width), 100), "\n")
    for rank, (sim, pos) in enumerate(zip(sims[i, :k], positions[i, :k]), start=1):
        neighbour = pool.iloc[pos]
        print(f"  #{rank}  similarity {sim:.3f}  [{LABEL_NAMES[neighbour.label]} | {neighbour.source}]")
        print(textwrap.indent(textwrap.fill(textwrap.shorten(neighbour.text, width), 96), "      "))
    print()

show_neighbours(0)

In [ ]:
show_neighbours(1)

Look at the `source` of each neighbour. Are the neighbours about the same **topic** as the query? Do they share its **label**?

## 7. Do neighbours share a label?

This is the question the whole retrieval idea depends on. If a text's neighbours tend to have the same label as the text itself, we can use them to predict the label.

Our pool is balanced (half human, half machine), so if neighbours were picked at random, about **50%** of them would share the query's label. Anything clearly above that means the embeddings carry some signal about who wrote the text.

In [ ]:
neighbour_labels = pool["label"].to_numpy()[positions]           # shape (200, k)
agreement = (neighbour_labels == queries["label"].to_numpy()[:, None]).mean()
print(f"Neighbours sharing the query's label: {agreement:.1%}  (random would be about 50%)")

## Check your understanding

<details>
<summary>1. Why do we take the pool and the queries from different MAGE splits?</summary>

If a query were also in the pool, its nearest neighbour would be itself, with similarity 1 and the correct label. The detector would look far better than it really is.
</details>

<details>
<summary>2. What does <code>IndexFlatIP</code> do, and when would you choose something else?</summary>

It compares the query with every stored vector using the inner (dot) product, so it is exact. For millions of vectors you would switch to an approximate index such as IVF or HNSW, trading a little accuracy for a lot of speed.
</details>

<details>
<summary>3. Neighbours share the query's label 50% of the time. What does that tell you?</summary>

That the embeddings carry no information about who wrote a text, at least as far as nearest neighbours are concerned. A retrieval-based detector built on them would be no better than flipping a coin.
</details>

## Try it yourself

1. Change `k` to 1, 15 and 50 and recompute the label agreement. What happens as you look further away?
2. Write a sentence of your own, embed it with `embedder.encode([...])`, and search the index. Which pool texts come back?
3. Try `faiss.IndexHNSWFlat(384, 32, faiss.METRIC_INNER_PRODUCT)` instead of `IndexFlatIP`. How often does it return the same neighbours?

## Summary

- Nearest-neighbour search finds the pool texts most similar to a query.
- With unit-length embeddings, brute force is a single matrix multiplication.
- **FAISS** does the same with an index, and scales to huge pools with approximate search.
- Retrieval can only help detection if neighbours tend to share the query's label.

**Next:** [Lesson 4 · Retrieval as a detector](04_retrieval_as_a_detector.ipynb), where we turn neighbours into predictions and measure how good they are.